TP 4-5-6 Casse brique
 
Le but de ce TP est de développer votre propre version du jeu classique Casse-brique (Breakout, 1976).
Le développement se fera en Python avec le module Tkinter pour l’interface graphique. Vous avez 3
séances de 4 heures, travail en binôme, et programmation orientée objet.
La notation portera autant sur la qualité du code (découpage en classes, lisibilité, commentaires
pertinents, utilisation de Git) que sur le résultat final.
Le but du jeu est de détruire tous les briques disposées en haut du canevas, à l’aide d’une balle
rebondissante renvoyée par une raquette contrôlée par le joueur. La partie est perdue si la balle tombe
en bas de l’écran.

Donc : POO, découpage en classe, bonnes pratiques, Git.





TO DO : 

In [ ]:
"""Function file TP4
Casse brique
Author: William GIRARDET
Date: October 8-22th 2026
ToDo: 
    Stocker les vies sous forme de pile
    Stocker les niveaux sous forme de file
    Matrice de briques correspondant à chaque niveau 
        0 --> espace vide
        1 --> brique classique 
        2 et + --> brique renforcée
        -1 --> pouvoir 
        -2 --> brique spéciale (effet)

    Menu : ouvrir une nouvelle fenêtre 
    - Resume
    - Restart current level / game
    - Level select
    - Quit

    Revoir le déplacement 

    "Each frame" :
    - check bar displacement 
    - check ball position 

    Create multiple zones in playing field:
    - bar / dead zone (could be separate or not)
    - empty zone (middle space where nothing happens)
    - game zone (main outline of where the bricks are, actually the brick matrix)
"""


import random   
import tkinter as tk

class Game:


class brick:
    def __init__(self, x, y, type):
        self.x = x
        self.y = y
        self.type = type
        self.is_destroyed = False

    def destroy(self):
        self.is_destroyed = True

    def is_destroyed(self):
        return self.is_destroyed



class Bille : 
    

Code Claude ci dessous :

In [ ]:

import random
import tkinter as tk
 
TAILLE = 500          # côté de l'espace de jeu (carré)
DELAI = 16            # ms entre deux images (~60 images/s)
 
 
class Plateforme:
    def __init__(self, canvas):
        self.canvas = canvas
        self.largeur = 90
        self.hauteur = 12
        self.vitesse = 8
        self.direction = 0  # -1 gauche, 0 immobile, +1 droite
        x = (TAILLE - self.largeur) / 2
        y = TAILLE - 30
        self.id = canvas.create_rectangle(x, y, x + self.largeur, y + self.hauteur,
                                          fill="white", outline="")
 
    def deplacer(self):
        x1, _, x2, _ = self.canvas.coords(self.id)
        dx = self.direction * self.vitesse
        # empêche de sortir de l'espace de jeu
        if x1 + dx < 0:
            dx = -x1
        elif x2 + dx > TAILLE:
            dx = TAILLE - x2
        self.canvas.move(self.id, dx, 0)
 
    def coords(self):
        return self.canvas.coords(self.id)
 
 
class Bille:
    def __init__(self, canvas):
        self.canvas = canvas
        self.rayon = 8
        self.id = canvas.create_oval(0, 0, 2 * self.rayon, 2 * self.rayon,
                                     fill="deepskyblue", outline="")
        self.reinitialiser()
 
    def reinitialiser(self):
        x = TAILLE / 2
        y = TAILLE / 2
        self.canvas.coords(self.id, x - self.rayon, y - self.rayon,
                           x + self.rayon, y + self.rayon)
        self.dx = random.choice([-4, 4])
        self.dy = -4
 
    def deplacer(self, plateforme):
        """Déplace la bille. Retourne False si elle tombe en bas."""
        self.canvas.move(self.id, self.dx, self.dy)
        x1, y1, x2, y2 = self.canvas.coords(self.id)
 
        # Rebonds sur les murs gauche, droite et haut
        if x1 <= 0:
            self.dx = abs(self.dx)
        elif x2 >= TAILLE:
            self.dx = -abs(self.dx)
        if y1 <= 0:
            self.dy = abs(self.dy)
 
        # Rebond sur la plateforme
        px1, py1, px2, py2 = plateforme.coords()
        if self.dy > 0 and y2 >= py1 and y1 <= py2 and x2 >= px1 and x1 <= px2:
            self.dy = -abs(self.dy)
            # l'angle dépend de l'endroit où la bille touche la plateforme
            centre_bille = (x1 + x2) / 2
            position = (centre_bille - (px1 + px2) / 2) / ((px2 - px1) / 2)  # -1 à 1
            self.dx = position * 6
            # petite accélération à chaque rebond
            self.dy = max(-9, self.dy * 1.03)
            return True, True
 
        # Bille perdue
        if y1 >= TAILLE:
            return False, False
        return True, False
 
 
class Jeu(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bille et plateforme")
        self.resizable(False, False)
 
        self.canvas = tk.Canvas(self, width=TAILLE, height=TAILLE,
                                bg="black", highlightthickness=0)
        self.canvas.pack()
 
        self.plateforme = Plateforme(self.canvas)
        self.bille = Bille(self.canvas)
 
        self.score = 0
        self.en_cours = True
        self.texte_score = self.canvas.create_text(10, 10, anchor="nw", fill="white",
                                                   font=("Arial", 14), text="Rebonds : 0")
        self.texte_fin = None
 
        # Contrôles : on mémorise les touches enfoncées pour un mouvement fluide
        self.touches = set()
        self.bind("<KeyPress-Left>", lambda e: self._touche(e, True))
        self.bind("<KeyPress-Right>", lambda e: self._touche(e, True))
        self.bind("<KeyRelease-Left>", lambda e: self._touche(e, False))
        self.bind("<KeyRelease-Right>", lambda e: self._touche(e, False))
        self.bind("<space>", lambda e: self.rejouer())
 
        self.boucle()
 
    def _touche(self, event, enfoncee):
        if enfoncee:
            self.touches.add(event.keysym)
        else:
            self.touches.discard(event.keysym)
        gauche = "Left" in self.touches
        droite = "Right" in self.touches
        self.plateforme.direction = (1 if droite else 0) - (1 if gauche else 0)
 
    def boucle(self):
        if self.en_cours:
            self.plateforme.deplacer()
            vivant, rebond = self.bille.deplacer(self.plateforme)
            if rebond:
                self.score += 1
                self.canvas.itemconfig(self.texte_score, text=f"Rebonds : {self.score}")
            if not vivant:
                self.fin_de_partie()
        self.after(DELAI, self.boucle)
 
    def fin_de_partie(self):
        self.en_cours = False
        self.texte_fin = self.canvas.create_text(
            TAILLE / 2, TAILLE / 2, fill="red", font=("Arial", 22, "bold"), justify="center",
            text=f"Perdu !\nScore : {self.score}\n\nEspace pour rejouer")
 
    def rejouer(self):
        if self.en_cours:
            return
        if self.texte_fin:
            self.canvas.delete(self.texte_fin)
            self.texte_fin = None
        self.score = 0
        self.canvas.itemconfig(self.texte_score, text="Rebonds : 0")
        self.bille.reinitialiser()
        self.en_cours = True
 
 
if __name__ == "__main__":
    Jeu().mainloop()
 
